# nb_orquestrador
Execução única por ciclo, chamada no fim do pipeline mestre:
1. Bronze → Silver: MERGE pela chave de negócio só das tabelas que receberam linhas neste lote
2. Gold: recalcula as tabelas Gold afetadas (notebooks cadastrados em `GOLD_NOTEBOOKS`, Fase 4)
3. Avança o `ultimo_watermark` **somente se tudo deu certo**
4. Grava o log do ciclo em `ctl_log`

Com `status_ciclo = "FALHA"` (caminho de erro do pipeline), só registra o log e não mexe em nada.

In [ ]:
lote = "manual"
ciclo = "FATOS"
resultados = "[]"       # lista "id|novo_watermark|linhas_copiadas", montada pelo pipeline
resultados_api = "[]"   # mesma lista, devolvida pelo nb_ingestao_api (origens API); "FALHOU" se ele quebrou
planejadas = "[]"       # ids das tabelas de banco que o nb_preparar_ciclo mandou copiar
status_ciclo = "SUCESSO"
mensagem = ""
origens = ""            # origens deste ciclo, ex.: "GS,4S" (vazio = todas): limita a retenção a elas
notificar = "SIM"       # aviso do ciclo no Teams; "NAO" quando chamado por uma rotina (ela faz o próprio)

In [ ]:
import os, json, time, datetime as dt
from pyspark.sql import SparkSession, DataFrame, Window, functions as F, types as T
from delta.tables import DeltaTable

# Não renomear os lakehouses: o banco do SQL endpoint guarda o nome da criação e a sincronização falha
LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

# processo e sessão em UTC: as datas da Bronze são horário de Brasília rotulado UTC e não podem ser
# deslocadas na leitura, no CAST para DATE nem na volta para o Python (collect devolve hora local)
os.environ["TZ"] = "UTC"
time.tzset()
spark = SparkSession.builder.getOrCreate()   # no Fabric a sessão já existe
spark.conf.set("spark.sql.session.timeZone", "UTC")
# coluna nova na origem entra no MERGE e no append sem recriar a tabela
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
    spark.sparkContext.setCheckpointDir(f"{_LOCAL}/_checkpoints")
else:
    import notebookutils
    # IDs, e não nomes, no endereço do OneLake: o ABFS do Spark não aceita os espaços do nome do
    # workspace ("Squad Dados - DW Dev")
    _WS = notebookutils.runtime.context["currentWorkspaceId"]
    _LH = {}
    def raiz(lakehouse):
        """abfss://<workspace>@onelake.../<lakehouse>: + /Tables/<tabela> ou + /Files/<caminho>."""
        if lakehouse not in _LH:
            item = notebookutils.lakehouse.get(lakehouse)
            _LH[lakehouse] = item["id"] if isinstance(item, dict) else item.id
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{_LH[lakehouse]}"
    def caminho(lakehouse, tabela):
        return f"{raiz(lakehouse)}/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)
    # resultados intermediários de materializar(), uma pasta por sessão (a rotina diária apaga as antigas)
    CHECKPOINTS = f"{raiz(LH_BRONZE)}/Files/_checkpoints"
    spark.sparkContext.setCheckpointDir(f"{CHECKPOINTS}/{spark.sparkContext.applicationId}")

def existe(lakehouse, tabela):
    try:
        return DeltaTable.isDeltaTable(spark, caminho(lakehouse, tabela))
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None, colunas=None):
    """DataFrame da tabela. filtros: [(coluna, operador, valor)], operadores =, >, >=, <, <=, in."""
    df = spark.read.format("delta").load(caminho(lakehouse, tabela))
    for coluna, op, valor in filtros or []:
        c = F.col(f"`{coluna}`")
        df = df.where(c.isin(list(valor)) if op == "in" else
                      {"=": c == valor, ">": c > valor, ">=": c >= valor, "<": c < valor, "<=": c <= valor}[op])
    return df.select(*[f"`{c}`" for c in colunas]) if colunas else df

def gravar(lakehouse, tabela, dados, modo="append"):
    """append acrescenta colunas novas (mergeSchema); overwrite troca dados e schema."""
    (dados.write.format("delta").mode(modo)
        .option("overwriteSchema" if modo == "overwrite" else "mergeSchema", "true")
        .save(caminho(lakehouse, tabela)))

def tabela_delta(lakehouse, tabela):
    return DeltaTable.forPath(spark, caminho(lakehouse, tabela))

def mesclar(lakehouse, tabela, dados, chaves):
    """MERGE pela chave: atualiza as linhas existentes e insere as novas (cria a tabela se não existir)."""
    if not existe(lakehouse, tabela):
        gravar(lakehouse, tabela, dados, modo="overwrite")
        return
    (tabela_delta(lakehouse, tabela).alias("t")
        .merge(dados.alias("s"), " AND ".join(f"t.`{c}` = s.`{c}`" for c in chaves))
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

def apagar(lakehouse, tabela, coluna, valores, extra=None):
    """DELETE das linhas com `coluna` em `valores`, em blocos (listas grandes de ids)."""
    alvo = tabela_delta(lakehouse, tabela)
    valores = list(valores)
    for i in range(0, len(valores), 5000):
        cond = F.col(f"`{coluna}`").isin(valores[i:i + 5000])
        alvo.delete(cond & F.expr(extra) if extra else cond)

def linhas_deletadas(lakehouse, tabela):
    """Linhas apagadas pela última operação (o DELETE do Spark não devolve métricas)."""
    m = tabela_delta(lakehouse, tabela).history(1).select("operationMetrics").first()[0] or {}
    return int(m.get("numDeletedRows") or 0)

def criar(linhas, schema):
    """DataFrame a partir de dicts. schema em DDL, ex.: "id string, n bigint, dt timestamp"."""
    return spark.createDataFrame(linhas, schema=schema, verifySchema=False)

def _utc(v):
    return v.replace(tzinfo=dt.timezone.utc) if isinstance(v, dt.datetime) and v.tzinfo is None else v

def coletar(df):
    """Linhas como dicts, com timestamp em UTC com fuso (o collect devolve sem fuso, em UTC)."""
    return [{k: _utc(v) for k, v in r.asDict().items()} for r in df.collect()]

def materializar(df):
    """Executa já e grava o resultado em arquivo: necessário antes de apagar ou regravar a tabela que o
    originou. Em arquivo, e não na memória (localCheckpoint): um executor desligado no meio da execução
    levava o resultado junto (CHECKPOINT_RDD_BLOCK_ID_NOT_FOUND)."""
    return df.checkpoint(eager=True)

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

def maximo(df, coluna):
    return _utc(df.agg(F.max(f"`{coluna}`")).first()[0])

def watermark_gold(gold, fonte):
    """Último _dt_carga da Silver já levado para esta Gold (tabela ctl_gold na Bronze). As Golds são
    incrementais por esta marca, e não pelo lote: se uma Gold falhar, a próxima execução retoma dali."""
    if not existe(LH_BRONZE, "ctl_gold"):
        return None
    r = ler(LH_BRONZE, "ctl_gold").where((F.col("gold") == gold) & (F.col("fonte") == fonte)).first()
    return _utc(r["ultimo_dt_carga"]) if r else None

def gravar_watermark_gold(gold, fonte, valor):
    linha = criar([{"gold": gold, "fonte": fonte, "ultimo_dt_carga": valor, "dt_atualizacao": agora_utc()}],
                  "gold string, fonte string, ultimo_dt_carga timestamp, dt_atualizacao timestamp")
    com_nova_tentativa(lambda: mesclar(LH_BRONZE, "ctl_gold", linha, ["gold", "fonte"]))

def com_nova_tentativa(acao, tentativas=5):
    """Execuções diferentes podem gravar a mesma tabela de controle ao mesmo tempo: o commit que perde a
    disputa é refeito sobre a versão nova da tabela."""
    for n in range(tentativas):
        try:
            return acao()
        except Exception as e:
            msg = str(e).lower()
            if n == tentativas - 1 or not any(p in msg for p in ("concurrent", "conflict", "commit")):
                raise
            time.sleep(5 * (n + 1))

def consulta(nome):
    """Texto de uma consulta da pasta `consultas` (no Fabric: lakehouse Gold → Files/consultas/<nome>.sql,
    copiada de implantacao/consultas pela implantação). Nos testes locais, lê de DW_CONSULTAS."""
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONSULTAS"], f"{nome}.sql"), encoding="utf-8") as arq:
            return arq.read()
    return notebookutils.fs.head(
        f"{raiz(LH_GOLD)}/Files/consultas/{nome}.sql", 10_000_000)

def cabecalho(sql):
    """Configuração no topo do .sql, em linhas `-- @chave valor` (ex.: `-- @destino gold_fato_coberturas`)."""
    cfg = {}
    for linha in sql.splitlines():
        linha = linha.strip()
        if linha.startswith("-- @"):
            chave, _, valor = linha[4:].partition(" ")
            cfg[chave] = valor.strip()
    return cfg

def ler_config(nome):
    """Arquivo YAML da pasta config (no Fabric: lakehouse Gold → Files/config/<nome>, copiado de config/ pela
    implantação). Nos testes locais, lê de DW_CONFIG."""
    import yaml
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONFIG"], nome), encoding="utf-8") as arq:
            return yaml.safe_load(arq)
    return yaml.safe_load(notebookutils.fs.head(
        f"{raiz(LH_GOLD)}/Files/config/{nome}", 10_000_000))

# ---------------------------------------------------------------- aviso no Teams por execução
# Resultado de cada ciclo (nb_orquestrador) e de cada rotina (nb_rotina_nexti) vai para o Teams pelo
# mesmo Workflow do nb_monitoramento. Config: config/qualidade.yml, seção `monitoramento.notificacao`.
_ICONE_STATUS = {"SUCESSO": "✅", "PARCIAL": "⚠️", "FALHA": "❌"}
_ICONE_COMP = {"NORMAL": "🟢", "ATENÇÃO": "🟡", "ALERTA": "🔴"}
_COR_COMP = {"NORMAL": "Good", "ATENÇÃO": "Warning", "ALERTA": "Attention"}

def _config_notificacao():
    return ((ler_config("qualidade.yml").get("monitoramento") or {}).get("notificacao")) or {}

def incrementar_contador(chave):
    """Contador persistente por chave (ctl_contador na Bronze); devolve o novo valor. Usado para avisar
    só 1 a cada N execuções das rotinas de alta frequência."""
    atual = 0
    if existe(LH_BRONZE, "ctl_contador"):
        r = ler(LH_BRONZE, "ctl_contador").where(F.col("chave") == chave).first()
        atual = int(r["n"]) if r else 0
    linha = criar([{"chave": chave, "n": atual + 1, "dt": agora_utc()}], "chave string, n bigint, dt timestamp")
    com_nova_tentativa(lambda: mesclar(LH_BRONZE, "ctl_contador", linha, ["chave"]))
    return atual + 1

def enviar_teams(corpo_cartao):
    """Posta um Adaptive Card no Workflow do Teams (URL no Key Vault, cofre e segredo em qualidade.yml).
    Não lança: uma notificação que falha não derruba a execução. Devolve o status do envio em texto."""
    notif = _config_notificacao()
    if _LOCAL:
        return "local"
    if not notif.get("teams_workflow_segredo") or not notif.get("key_vault"):
        return "sem workflow"
    try:
        import requests
        url = notebookutils.credentials.getSecret(notif["key_vault"], notif["teams_workflow_segredo"])
        payload = {"type": "message", "destinatario": notif.get("destinatario"),
                   "attachments": [{"contentType": "application/vnd.microsoft.card.adaptive", "content": {
                       "$schema": "http://adaptivecards.io/schemas/adaptive-card.json", "type": "AdaptiveCard",
                       "version": "1.4", "body": corpo_cartao}}]}
        r = requests.post(url, json=payload, timeout=30)
        r.raise_for_status()
        return f"enviado para {notif.get('destinatario') or 'o Workflow'}"
    except Exception as e:   # sem Workflow configurado ou fora do ar: a execução segue normal
        return f"não notificado: {str(e)[:200]}"

def comportamento_execucao(status, duracao_s, limite_atencao=None):
    """🔴 ALERTA: a base não atualizou (FALHA/PARCIAL). 🟡 ATENÇÃO: concluiu, mas acima do tempo esperado
    (`limite_atencao`). 🟢 NORMAL: concluiu no tempo normal."""
    if status in ("FALHA", "PARCIAL"):
        return "ALERTA"
    if limite_atencao and duracao_s and duracao_s > limite_atencao:
        return "ATENÇÃO"
    return "NORMAL"

def avisar_execucao(base, objeto, status, duracao_s, erro="", resumo="", tipo="CICLO", chave=None):
    """Aviso no Teams do resultado de uma execução (ciclo do pipeline ou rotina). As rotinas da Nexti só
    avisam o sucesso NORMAL 1 vez a cada `nexti_cada` execuções (config); falha, PARCIAL e ATENÇÃO avisam
    sempre. Devolve o status do envio ('enviado...', 'local', 'agrupado', 'desligado', 'sem workflow')."""
    notif = _config_notificacao()
    if not notif.get("avisar_execucoes", True):
        return "desligado"
    comp = comportamento_execucao(status, duracao_s, (notif.get("duracao_atencao_s") or {}).get(tipo))
    cada = int(notif.get("nexti_cada", 1)) if str(base).upper() == "NEXTI" else 1
    if comp == "NORMAL" and cada > 1 and chave and incrementar_contador(f"AVISO_{chave}") % cada != 1:
        return "agrupado"
    fatos = [("BASE", base), ("Notebook/Pipeline", objeto),
             ("Status", f"{_ICONE_STATUS.get(status, '•')} {status}")]
    if erro:
        fatos.append(("Erro", erro[:800]))
    fatos.append(("Horário", agora_utc().astimezone(FUSO_BRT).strftime("%d/%m/%Y %H:%M")))
    fatos.append(("Comportamento", f"{_ICONE_COMP[comp]} {comp}"))
    corpo = [{"type": "TextBlock", "size": "Medium", "weight": "Bolder", "color": _COR_COMP[comp],
              "text": f"{_ICONE_COMP[comp]} DW Fabric — {comp}"}]
    if resumo:
        corpo.append({"type": "TextBlock", "isSubtle": True, "wrap": True, "spacing": "None", "text": resumo[:300]})
    corpo.append({"type": "FactSet", "facts": [{"title": t, "value": str(v)} for t, v in fatos]})
    return enviar_teams(corpo)

In [ ]:
# Tabela Silver -> lista de notebooks Gold a executar quando ela muda (preencher na Fase 4)
GOLD_NOTEBOOKS = {
    # "gs_lan": ["nb_gold_f_lancamentos"],
}

# Retenção da Bronze (dias de lotes mantidos), aplicada só no ciclo COMPLETO.
# FULL: cada lote é uma cópia inteira, basta guardar poucos dias para reprocessar.
RETENCAO_DIAS = {"FULL": 3, "JANELA": 3, "INCREMENTAL": 30}

INICIO = agora_utc()
_log = []
FILTRO = {o.strip().upper() for o in origens.split(",") if o.strip()}

SCHEMA_LOG = ("lote string, ciclo string, tabela_id string, etapa string, linhas bigint, status string, "
              "mensagem string, dt_registro timestamp")

def log(tabela_id, etapa, linhas, status, msg=""):
    _log.append({"lote": lote, "ciclo": ciclo, "tabela_id": tabela_id, "etapa": etapa,
                 "linhas": int(linhas or 0), "status": status, "mensagem": (msg or "")[:4000],
                 "dt_registro": agora_utc()})

def gravar_log():
    if _log:
        com_nova_tentativa(lambda: gravar(LH_BRONZE, "ctl_log", criar(_log, SCHEMA_LOG)))

def interpretar(resultados):
    """Itens "id|novo_watermark|linhas[|mensagem]". novo_watermark = FALHA (cópia falhou) ou SEM_MUDANCA
    (incremental sem alteração na origem, cópia pulada)."""
    itens = []
    for s in json.loads(resultados or "[]"):
        tid, novo, linhas, msg = (s.split("|", 3) + ["", "", "0", ""])[:4]
        itens.append({"id": tid, "novo": novo, "linhas": int(float(linhas or 0)), "msg": msg})
    return itens

def q(nome):
    return "`" + nome.replace("`", "``") + "`"

def padronizar(df):
    """Regras da Silver (docs/02_levantamento_silver.md): float -> DECIMAL(18,6), com NaN/infinito
    -> NULL; texto sem espaços no fim e vazio -> NULL; binários removidos. Colunas técnicas (_*) e
    demais tipos ficam como estão. Devolve (DataFrame, aviso com as conversões de NaN/infinito)."""
    flutuantes = [f.name for f in df.schema if not f.name.startswith("_")
                  and isinstance(f.dataType, (T.FloatType, T.DoubleType))]
    def nao_finito(c):
        return F.isnan(F.col(q(c))) | (F.abs(F.col(q(c))) == float("inf"))
    nao_finitos = []
    if flutuantes:
        contagem = df.select([F.sum(F.when(nao_finito(c), 1).otherwise(0)).alias(c) for c in flutuantes]).first()
        nao_finitos = [f"{c}={contagem[c]}" for c in flutuantes if contagem[c]]
    exprs = []
    for f in df.schema:
        c = F.col(q(f.name))
        if f.name.startswith("_"):
            exprs.append(c)
        elif f.name in flutuantes:
            exprs.append(F.when(~nao_finito(f.name), c).cast("decimal(18,6)").alias(f.name))
        elif isinstance(f.dataType, T.StringType):
            exprs.append(F.when(F.rtrim(c) != "", F.rtrim(c)).alias(f.name))
        elif not isinstance(f.dataType, T.BinaryType):
            exprs.append(c)
    return df.select(exprs), ("NaN/infinito -> NULL: " + ", ".join(nao_finitos)) if nao_finitos else ""

def carregar_silver(cfg, linhas_copiadas):
    """Leva o lote da Bronze para a Silver. Devolve quantas linhas foram gravadas."""
    destino = nome_silver(cfg["tabela_destino"])
    if not existe(LH_BRONZE, nome_bronze(cfg["tabela_destino"])):  # origem API sem nenhuma linha ainda
        return 0, ""
    bruto = ler(LH_BRONZE, nome_bronze(cfg["tabela_destino"]), filtros=[("_lote", "=", lote)])
    if bruto.isEmpty():
        return 0, ""
    # nomes sem diferenciar maiúsculas: o Firebird (4S) devolve EMP_CODIGO para EMP_Codigo
    por_nome = {c.upper(): c for c in bruto.columns}
    chaves = [por_nome.get(c.strip().upper(), c.strip()) for c in cfg["chave_negocio"].split(",") if c.strip()]
    faltando = [c for c in chaves if c not in bruto.columns]
    if faltando:
        raise ValueError(f"chave_negocio não encontrada na Bronze: {faltando}")
    ordem = por_nome.get((cfg["coluna_watermark"] or "").upper(), "_dt_carga")
    janela = Window.partitionBy(*[q(c) for c in chaves]).orderBy(F.col(q(ordem)).desc_nulls_last())
    unico = (bruto.withColumn("_rn", F.row_number().over(janela)).where("_rn = 1").drop("_rn")
             .withColumn("_ativo", F.lit(True)).withColumn("_dt_atualizacao", F.current_timestamp()))
    unico, aviso = padronizar(unico)
    unico = materializar(unico)
    n = unico.count()

    if cfg["modo_carga"] == "FULL" or not existe(LH_SILVER, destino):
        # dimensões: retrato completo do dia substitui a versão anterior
        gravar(LH_SILVER, destino, unico, modo="overwrite")
    else:
        mesclar(LH_SILVER, destino, unico, chaves)
    return n, aviso

def aplicar_retencao(ctl):
    """Apaga da Bronze os lotes mais antigos que RETENCAO_DIAS, compacta os arquivos e remove
    fisicamente o que não é mais referenciado (após 7 dias). Falha aqui não derruba o ciclo."""
    for cfg in ctl.values():
        nome = nome_bronze(cfg["tabela_destino"])
        # só as origens deste ciclo: outra origem pode estar gravando a Bronze em paralelo
        if FILTRO and cfg["origem"] not in FILTRO:
            continue
        if not cfg["ativo"] or not existe(LH_BRONZE, nome):
            continue
        limite = agora_utc() - dt.timedelta(days=RETENCAO_DIAS.get(cfg["modo_carga"], 30))
        try:
            alvo = tabela_delta(LH_BRONZE, nome)
            alvo.delete(F.col("_dt_carga") < F.lit(limite))
            apagadas = linhas_deletadas(LH_BRONZE, nome)
            alvo.optimize().executeCompaction()
            alvo.vacuum(168)
            log(cfg["id"], "RETENCAO", apagadas, "SUCESSO")
        except Exception as e:
            log(cfg["id"], "RETENCAO", 0, "FALHA", str(e))

def rodar_gold(tabelas_alteradas):
    for nb in dict.fromkeys(n for t in tabelas_alteradas for n in GOLD_NOTEBOOKS.get(t, [])):
        notebookutils.notebook.run(nb, 1800, {"lote": lote})
        log(nb, "GOLD", 0, "SUCESSO")

def avancar_watermarks(itens, ctl):
    agora = agora_utc()
    novos = [{"id": i["id"], "novo": i["novo"], "dt": agora} for i in itens
             if i["novo"] and ctl[i["id"]]["modo_carga"] == "INCREMENTAL"]
    if not novos:
        return
    fonte = criar(novos, "id string, novo string, dt timestamp")
    com_nova_tentativa(lambda: tabela_delta(LH_BRONZE, "ctl_tabelas").alias("t")
        .merge(fonte.alias("s"), "t.id = s.id")
        .whenMatchedUpdate(set={"ultimo_watermark": "s.novo", "dt_atualizacao": "s.dt"})
        .execute())

In [ ]:
if status_ciclo != "SUCESSO":
    log("CICLO", "CICLO", 0, "FALHA", mensagem or "Falha em etapa do pipeline")
    gravar_log()
    if notificar == "SIM":
        avisar_execucao(origens or "todas", f"Ciclo {ciclo} (pl_mestre_dw)", "FALHA",
                        (agora_utc() - INICIO).total_seconds(),
                        erro=mensagem or "Falha antes do orquestrador", tipo="CICLO")
    sair("FALHA registrada")
else:
    # Cada tabela é independente: a que falhou fica com o watermark parado e vai para o log; as outras
    # seguem para a Silver normalmente. No fim, havendo falha, o notebook falha (pipeline vermelho).
    falhas = []
    itens = interpretar(resultados)
    if resultados_api == "FALHOU":
        falhas.append("Ingestao_API: notebook falhou antes de devolver os resultados")
        log("Ingestao_API", "BRONZE", 0, "FALHA", "notebook nb_ingestao_api falhou")
    else:
        itens += interpretar(resultados_api)
    ctl = {r["id"]: r for r in coletar(ler(LH_BRONZE, "ctl_tabelas"))}
    reportadas = {i["id"] for i in itens}
    for tid in json.loads(planejadas or "[]"):
        if tid not in reportadas:   # o ForEach não chegou a registrar: a cópia falhou
            falhas.append(f"{tid}: cópia não concluída")
            log(tid, "BRONZE", 0, "FALHA", "cópia não concluída (ver a execução do pipeline)")
    ok, alteradas = [], []
    for i in itens:
        cfg = ctl.get(i["id"])
        if cfg is None:
            falhas.append(f"{i['id']}: não está em ctl_tabelas")
            continue
        if i["novo"] == "FALHA":
            falhas.append(f"{i['id']}: {i['msg']}")
            log(i["id"], "BRONZE", 0, "FALHA", i["msg"])
            continue
        if i["novo"] == "SEM_MUDANCA":
            continue
        log(i["id"], "BRONZE", i["linhas"], "SUCESSO")
        try:
            n, aviso = carregar_silver(cfg, i["linhas"])
            log(i["id"], "SILVER", n, "SUCESSO", aviso)
            ok.append(i)
            if n:
                alteradas.append(cfg["tabela_destino"])
        except Exception as e:
            falhas.append(f"{i['id']}: {e}")
            log(i["id"], "SILVER", 0, "FALHA", str(e))

    try:
        rodar_gold(alteradas)
    except Exception as e:
        falhas.append(f"GOLD: {e}")
        log("GOLD", "GOLD", 0, "FALHA", str(e))

    avancar_watermarks(ok, ctl)              # só as tabelas que chegaram à Silver
    if ciclo == "COMPLETO":
        aplicar_retencao(ctl)
    duracao = (agora_utc() - INICIO).total_seconds()
    status = "SUCESSO" if not falhas else ("PARCIAL" if ok else "FALHA")
    log("CICLO", "CICLO", sum(i["linhas"] for i in ok), status,
        f"Origens {origens or 'todas'}: {len(ok)} tabelas na Silver, {len(alteradas)} alteradas, "
        f"{len(falhas)} com falha, {duracao:.0f}s no notebook" + (" | " + " | ".join(falhas) if falhas else ""))
    gravar_log()
    if notificar == "SIM":
        avisar_execucao(origens or "todas", f"Ciclo {ciclo} (pl_mestre_dw)", status, duracao,
                        erro=" | ".join(falhas),
                        resumo=f"{len(ok)} tabelas na Silver, {len(alteradas)} Gold(s) afetada(s)", tipo="CICLO")
    if falhas:
        raise RuntimeError(f"Ciclo {status}: {len(ok)} tabelas atualizadas; watermark parado nas que falharam:\n"
                           + "\n".join(falhas))
    sair(json.dumps({"tabelas": len(ok), "alteradas": alteradas}))